# Análise Exploratória de Dados (EDA) — datathon-fraude

**Projeto:** Plataforma MLOps para Prevenção a Fraude  
**Autora:** Maria L F de Araujo — Turma 6MLET, FIAP/PosTech  
**Dataset:** enriched-v2 (10.000 transações sintéticas, seed=42)  
**Objetivo:** Explorar a distribuição dos dados, identificar padrões de fraude e justificar as decisões de feature engineering.

---

## Estrutura do Notebook

1. Carregamento e visão geral do dataset
2. Distribuição das classes (fraude vs legítimo)
3. Análise das features principais
4. Correlação com `is_fraud`
5. Distribuição por `merchant_category`
6. Insights e decisões de negócio


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
sns.set_palette("husl")

print("✅ Imports carregados")


## 1. Carregamento e Visão Geral

In [ ]:
# Carrega datasets
df_raw = pd.read_csv('../data/raw/transactions.csv')
df_feat = pd.read_csv('../data/processed/features.csv')

# Merge para análise completa
df = df_feat.merge(df_raw[['transaction_id', 'merchant_category', 'timestamp', 'channel', 'city']], 
                   on='transaction_id', how='left')

print(f"Shape: {df.shape}")
print(f"\nColunas: {df.columns.tolist()}")
print(f"\nTipos:")
print(df.dtypes)


In [ ]:
# Visão geral estatística
print("=== Estatísticas Descritivas ===")
df.describe().round(3)


In [ ]:
# Verificação de nulos
print("=== Valores Nulos ===")
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else "✅ Nenhum valor nulo encontrado")


## 2. Distribuição das Classes (Fraude vs Legítimo)

In [ ]:
# Distribuição de classes
fraud_count = df['is_fraud'].value_counts()
fraud_pct = df['is_fraud'].value_counts(normalize=True) * 100

print("=== Distribuição de Classes ===")
print(f"Legítimas: {fraud_count[0]:,} ({fraud_pct[0]:.1f}%)")
print(f"Fraudes:   {fraud_count[1]:,} ({fraud_pct[1]:.1f}%)")
print(f"\nRatio de desbalanceamento: {fraud_count[0]/fraud_count[1]:.0f}:1")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pizza
axes[0].pie(fraud_count, labels=['Legítima', 'Fraude'], autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90, textprops={'fontsize': 13})
axes[0].set_title('Distribuição de Classes', fontsize=14, fontweight='bold')

# Barras
bars = axes[1].bar(['Legítima', 'Fraude'], fraud_count.values, 
                   color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
axes[1].set_title('Contagem por Classe', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Número de Transações')
for bar, val in zip(bars, fraud_count.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
                f'{val:,}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../docs/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n⚠️ Desbalanceamento severo (98:2) — justifica uso de scale_pos_weight no XGBoost")


## 3. Análise das Features Principais

In [ ]:
# Feature importance do modelo
import joblib
model = joblib.load('../models/champion_v3.joblib')

FEATURE_COLS = ['amount', 'distance_from_home', 'velocity_1h', 'velocity_24h',
    'avg_amount_30d', 'account_balance', 'is_new_device', 'time_since_last_txn_min',
    'failed_txns_last_24h', 'ip_risk_score', 'amount_ratio', 'is_night',
    'high_velocity', 'is_online', 'is_credit', 'is_urgent', 'merchant_category_encoded']

importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c' if imp > 0.1 else '#f39c12' if imp > 0.01 else '#3498db' 
          for imp in importances.values]
bars = ax.barh(importances.index, importances.values, color=colors)
ax.set_xlabel('Importância (gain)', fontsize=12)
ax.set_title('Feature Importance — XGBoost Champion v3', fontsize=14, fontweight='bold')

# Legenda
from matplotlib.patches import Patch
legend = [Patch(color='#e74c3c', label='Alto Impacto (>0.1)'),
          Patch(color='#f39c12', label='Moderado (0.01-0.1)'),
          Patch(color='#3498db', label='Contextual (<0.01)')]
ax.legend(handles=legend, loc='lower right')

plt.tight_layout()
plt.savefig('../docs/eda_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== Top Features ===")
for feat, imp in importances.sort_values(ascending=False).head(5).items():
    print(f"  {feat:<30} {imp:.4f}")


In [ ]:
# Distribuição das top 4 features por classe
top_features = ['time_since_last_txn_min', 'ip_risk_score', 'velocity_24h', 'distance_from_home']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feature in enumerate(top_features):
    fraud = df[df['is_fraud'] == 1][feature]
    legit = df[df['is_fraud'] == 0][feature]
    
    axes[i].hist(legit, bins=50, alpha=0.6, color='#2ecc71', label='Legítima', density=True)
    axes[i].hist(fraud, bins=50, alpha=0.6, color='#e74c3c', label='Fraude', density=True)
    axes[i].set_title(f'Distribuição: {feature}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Densidade')
    axes[i].legend()

plt.suptitle('Distribuição das Top Features por Classe', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../docs/eda_top_features_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Correlação com `is_fraud`

In [ ]:
# Correlação de todas as features numéricas com is_fraud
numeric_cols = df[FEATURE_COLS + ['is_fraud']].select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()['is_fraud'].drop('is_fraud').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c' if c > 0 else '#3498db' for c in corr.values]
bars = ax.barh(corr.index, corr.values, color=colors)
ax.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Correlação de Pearson com is_fraud', fontsize=12)
ax.set_title('Correlação das Features com Fraude', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../docs/eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== Top Correlações com is_fraud ===")
print("\nMais correlacionadas (positivo = indica fraude):")
for feat, corr_val in corr.head(5).items():
    print(f"  {feat:<30} {corr_val:.4f}")
print("\nMenos correlacionadas:")
for feat, corr_val in corr.tail(3).items():
    print(f"  {feat:<30} {corr_val:.4f}")


## 5. Distribuição por `merchant_category`

In [ ]:
# Taxa de fraude por merchant_category
cat_stats = df.groupby('merchant_category').agg(
    total=('is_fraud', 'count'),
    fraudes=('is_fraud', 'sum')
).assign(taxa_fraude=lambda x: x['fraudes'] / x['total'] * 100).sort_values('taxa_fraude', ascending=False)

print("=== Taxa de Fraude por Categoria ===")
print(cat_stats.round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Taxa de fraude
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(cat_stats)))
bars = axes[0].bar(cat_stats.index, cat_stats['taxa_fraude'], color=colors, edgecolor='white')
axes[0].set_title('Taxa de Fraude por Categoria (%)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Taxa de Fraude (%)')
axes[0].set_xticklabels(cat_stats.index, rotation=30, ha='right')
for bar, val in zip(bars, cat_stats['taxa_fraude']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{val:.1f}%', ha='center', fontsize=10)

# Volume total
axes[1].bar(cat_stats.index, cat_stats['total'], color='#3498db', 
            edgecolor='white', label='Total')
axes[1].bar(cat_stats.index, cat_stats['fraudes'], color='#e74c3c', 
            edgecolor='white', label='Fraudes')
axes[1].set_title('Volume de Transações por Categoria', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Número de Transações')
axes[1].set_xticklabels(cat_stats.index, rotation=30, ha='right')
axes[1].legend()

plt.tight_layout()
plt.savefig('../docs/eda_merchant_category.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Fairness — Recall por merchant_category
df['pred'] = (model.predict_proba(df[FEATURE_COLS])[:, 1] >= 0.5).astype(int)

fairness = []
for cat, group in df.groupby('merchant_category'):
    frauds = group[group['is_fraud'] == 1]
    if len(frauds) == 0:
        continue
    recall = (frauds['pred'] == 1).sum() / len(frauds)
    precision = group[(group['pred']==1)&(group['is_fraud']==1)].shape[0] / max(group[group['pred']==1].shape[0], 1)
    fairness.append({'Categoria': cat, 'N° Fraudes': len(frauds), 
                     'Recall': round(recall, 4), 'Precision': round(precision, 4)})

fairness_df = pd.DataFrame(fairness)
print("=== Análise de Fairness por merchant_category ===")
print(fairness_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(fairness_df))
bars = ax.bar(fairness_df['Categoria'], fairness_df['Recall'], 
              color='#2ecc71', edgecolor='white', label='Recall')
ax.bar(fairness_df['Categoria'], fairness_df['Precision'], 
       alpha=0.5, color='#3498db', edgecolor='white', label='Precision')
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Recall=1.0 (global)')
ax.set_ylim(0, 1.1)
ax.set_title('Fairness — Recall e Precision por Categoria', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.legend()

plt.tight_layout()
plt.savefig('../docs/eda_fairness.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Recall 1.0 em todas as categorias — ausência de disparidade")


## 6. Insights e Decisões de Negócio

### 6.1 Principais Achados

| Insight | Impacto | Decisão Tomada |
|---|---|---|
| Desbalanceamento severo (98:2) | Alto | `scale_pos_weight` no XGBoost para compensar |
| `time_since_last_txn_min` domina (93.67%) | Alto | Feature de engenharia temporal como foco principal |
| `is_urgent` correlação 0.770 com fraude | Alto | Feature engenharia combinando device_novo + tempo |
| `ip_risk_score > 0.7` forte indicador | Médio | Regra de negócio indexada no RAG |
| Recall 1.0 em todas as categorias | Positivo | Ausência de viés por merchant_category confirmada |
| Fraudes concentradas em varejo (75) e entretenimento (62) | Médio | Regras específicas para esses segmentos no RAG |

### 6.2 Decisão de Priorizar Recall

O custo de um falso negativo (fraude não detectada = prejuízo financeiro real) é **significativamente maior** que o custo de um falso positivo (transação legítima bloqueada = atrito com cliente, reversível). Esta assimetria justifica a priorização de Recall sobre Precision e o uso de `scale_pos_weight`.

### 6.3 Feature Engineering

As features mais importantes foram derivadas de comportamento temporal:
- `time_since_last_txn_min` — tempo desde a última transação (dominante)
- `is_urgent` — combinação de device_novo + transação rápida (correlação 0.770)
- `velocity_24h` — número de transações nas últimas 24h (card testing)

Essas features capturam padrões de **account takeover** e **card testing** — os dois tipos de fraude mais frequentes no dataset.


In [ ]:
print("✅ EDA concluído!")
print("\nArquivos gerados em docs/:")
import os
eda_files = [f for f in os.listdir('../docs') if f.startswith('eda_')]
for f in eda_files:
    print(f"  - {f}")
